#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType


In [0]:

RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_create_date"
}

#Read data from bronze

In [0]:
df = spark.table("data_lakehouse_project.bronze.bronze_cust_info")

#Data Transform

In [0]:
#trim all string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
#Normalize information cst_marital_status,cst_gndr
df = (
    df
    .withColumn("cst_marital_status",
    F.when(F.upper(col("cst_marital_status")) == "M","Married")
    .when(F.upper(col("cst_marital_status")) == "S","Single")
    .otherwise("n/a")
)
)
df = (
    df
    .withColumn("cst_gndr",
    F.when(F.upper(col("cst_gndr")) == "M","Male")
    .when(F.upper(col("cst_gndr")) == "F","Female")
    .otherwise("n/a")
)
)

In [0]:
# Renaming columns header
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)


#Write to Silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("data_lakehouse_project.silver.silver_cust_info")
)